# Maternal Health Risk Prediction

## Data Preprocessing

#### Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings("ignore")

#### Load Dataset

In [2]:
data = pd.read_csv("../Dataset/Maternal Health Risk Data Set.csv")
data.head()

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,25,130,80,15.0,98.0,86,high risk
1,35,140,90,13.0,98.0,70,high risk
2,29,90,70,8.0,100.0,80,high risk
3,30,140,85,7.0,98.0,70,high risk
4,35,120,60,6.1,98.0,76,low risk


#### Dataset Information

In [3]:
print(data.shape)
data.info()

(1014, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          1014 non-null   int64  
 1   SystolicBP   1014 non-null   int64  
 2   DiastolicBP  1014 non-null   int64  
 3   BS           1014 non-null   float64
 4   BodyTemp     1014 non-null   float64
 5   HeartRate    1014 non-null   int64  
 6   RiskLevel    1014 non-null   object 
dtypes: float64(2), int64(4), object(1)
memory usage: 55.6+ KB


#### Missing Value Verification

In [4]:
data.isnull().sum()

Age            0
SystolicBP     0
DiastolicBP    0
BS             0
BodyTemp       0
HeartRate      0
RiskLevel      0
dtype: int64

#### Label Encoding Target Variable

In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data["RiskLevel_Encoded"] = le.fit_transform(data["RiskLevel"])

data.head()

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel,RiskLevel_Encoded
0,25,130,80,15.0,98.0,86,high risk,0
1,35,140,90,13.0,98.0,70,high risk,0
2,29,90,70,8.0,100.0,80,high risk,0
3,30,140,85,7.0,98.0,70,high risk,0
4,35,120,60,6.1,98.0,76,low risk,1


#### Label Encoder Class Mapping

In [6]:
print("Classes :", le.classes_)

Classes : ['high risk' 'low risk' 'mid risk']


#### Save Label Encoder

In [7]:
with open("../Model/risklevel_encoder.pkl","wb") as f:
    pickle.dump(le,f)

#### Feature Target Selection

In [8]:
X = data.drop(
    columns=[
        "RiskLevel",
        "RiskLevel_Encoded"
    ]
)

y = data["RiskLevel_Encoded"]

print(X.shape)
print(y.shape)

(1014, 6)
(1014,)


#### Train Validation Test Split (70-15-15)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

#### Stratified Data Split Verification

In [10]:
print("="*50)
print("Training Distribution (%)")
print(round(y_train.value_counts(normalize=True)*100,2))

print("\n"+"="*50)
print("Validation Distribution (%)")
print(round(y_val.value_counts(normalize=True)*100,2))

print("\n"+"="*50)
print("Testing Distribution (%)")
print(round(y_test.value_counts(normalize=True)*100,2))

print("\n"+"="*50)
print("Dataset Shapes")
print("="*50)

print("X_train :", X_train.shape)
print("X_val   :", X_val.shape)
print("X_test  :", X_test.shape)

Training Distribution (%)
RiskLevel_Encoded
1    40.06
2    33.15
0    26.80
Name: proportion, dtype: float64

Validation Distribution (%)
RiskLevel_Encoded
1    40.13
2    32.89
0    26.97
Name: proportion, dtype: float64

Testing Distribution (%)
RiskLevel_Encoded
1    39.87
2    33.33
0    26.80
Name: proportion, dtype: float64

Dataset Shapes
X_train : (709, 6)
X_val   : (152, 6)
X_test  : (153, 6)


#### Feature Scaling

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

#### Save Scaler

In [12]:
with open("../Model/scaler.pkl","wb") as f:
    pickle.dump(scaler,f)

#### Save Processed Files

In [13]:
pd.DataFrame(
    X_train_scaled,
    columns=X.columns
).to_csv("../Dataset/X_train.csv",index=False)

pd.DataFrame(
    X_val_scaled,
    columns=X.columns
).to_csv("../Dataset/X_val.csv",index=False)

pd.DataFrame(
    X_test_scaled,
    columns=X.columns
).to_csv("../Dataset/X_test.csv",index=False)

y_train.to_csv("../Dataset/y_train.csv",index=False)
y_val.to_csv("../Dataset/y_val.csv",index=False)
y_test.to_csv("../Dataset/y_test.csv",index=False)